DAY 23 ADAVANCED FILTERING AND CONDITIONAL ANALYSIS

In [47]:
import pandas as pd
import numpy as np

In [48]:
df = pd.read_csv("Data_Co_Supply_Chain_Dataset.csv" , encoding="latin1")
df.shape

(180519, 31)

## Single-Condition Filtering

In [49]:
## Which orders have sales greater than 500?
high_sales_orders = df[df["Sales"] > 500]
high_sales_orders.shape

(972, 31)

In [50]:
high_sales_orders[["Order Id" , "Sales" , "Customer Segment" , "Order Region"]].head()

,Order Id,Sales,Customer Segment,Order Region
152,70734,1500.00000,Consumer,Northern Europe
155,70769,1500.00000,Consumer,Northern Europe
160,70534,1500.00000,Consumer,Western Europe
172,70544,1500.00000,Consumer,Northern Europe
174,68879,999.98999,Consumer,Western Europe


## Multi-Condition Filtering

In [51]:
# Find orders where Sales > 500 AND the customer is in the Consumer segment.
consumer_high_sales = df[(df["Sales"] > 500) & (df["Customer Segment"] == "Consumer")]
consumer_high_sales.shape

(520, 31)

In [52]:
consumer_high_sales[["Order Id" , "Sales" , "Customer Segment" ,"Order Region"]].head(10)

,Order Id,Sales,Customer Segment,Order Region
152,70734,1500.000000,Consumer,Northern Europe
155,70769,1500.000000,Consumer,Northern Europe
160,70534,1500.000000,Consumer,Western Europe
172,70544,1500.000000,Consumer,Northern Europe
174,68879,999.989990,Consumer,Western Europe
1594,70795,1500.000000,Consumer,Western Europe
3495,74536,1500.000000,Consumer,South Asia
3496,74570,1500.000000,Consumer,Eastern Asia
3623,75144,532.580017,Consumer,Eastern Asia
3627,75131,532.580017,Consumer,South Asia


## OR Condition + isin()

In [53]:
# Find orders from either Consumer OR Corporate customers where Sales > 500.
consumer_corporate_high_sales = df[
     (df["Sales"] > 500) &
     (df["Customer Segment"].isin(["Consumer" , "Corporate"]))
]
consumer_corporate_high_sales.shape

(828, 31)

In [54]:
consumer_corporate_high_sales[["Order Id" , "Sales" , "Customer Segment" , "Order Region"]].head(10)

,Order Id,Sales,Customer Segment,Order Region
152,70734,1500.000000,Consumer,Northern Europe
155,70769,1500.000000,Consumer,Northern Europe
160,70534,1500.000000,Consumer,Western Europe
172,70544,1500.000000,Consumer,Northern Europe
174,68879,999.989990,Consumer,Western Europe
1594,70795,1500.000000,Consumer,Western Europe
2325,71986,532.580017,Corporate,Oceania
2328,71894,532.580017,Corporate,Eastern Asia
2330,71947,532.580017,Corporate,Southeast Asia
2331,71821,532.580017,Corporate,Southeast Asia


## between()

In [55]:
# Find transactions where Sales is between 100 and 500.
medium_sales_orders = df[
     df["Sales"].between(100 , 500)
]
medium_sales_orders.shape

(144631, 31)

In [56]:
medium_sales_orders["Sales"].agg(["min" , "max"])

min    100.0
max    500.0
Name: Sales, dtype: float64

## Combine Multiple Filtering Techniques

In [57]:
## Find Consumer or Corporate transactions where Sales is between 100 and 500, and the order has no late-delivery risk.
target_orders = df[
     (df["Sales"].between(100 , 500)) &
     (df["Customer Segment"].isin(["Consumer" , "Corporate"])) &
     (df["Late_delivery_risk"] == 0)
]
target_orders.shape

(53917, 31)

In [58]:
target_orders[
     ["Order Id" , "Sales" , "Customer Segment" , "Late_delivery_risk" , "Order Region"]
].head(10)

,Order Id,Sales,Customer Segment,Late_delivery_risk,Order Region
0,77202,327.75,Consumer,0,Southeast Asia
2,75938,327.75,Consumer,0,South Asia
4,75936,327.75,Corporate,0,Oceania
5,75935,327.75,Consumer,0,Oceania
10,75930,327.75,Corporate,0,Eastern Asia
19,75921,327.75,Consumer,0,South Asia
20,75920,327.75,Corporate,0,South Asia
23,75917,327.75,Corporate,0,Oceania
24,75916,327.75,Corporate,0,Oceania
27,75913,327.75,Corporate,0,Eastern Asia


FILTERING BUSINESS CHALLENGE

In [59]:
high_value_transactions = df[
     (df["Sales"] > 500) & 
     (df["Customer Segment"].isin(["Consumer" , "Corporate"])) &
     (df["Delivery Status"]== "Late delivery") &
     (df["Order Region"].isin(["Western Europe" , "Central America" , "Southern Europe"]))
]
high_value_transactions.shape

(134, 31)

In [60]:
high_value_transactions[
     ["Order Id" , "Sales" , "Customer Segment" , "Delivery Status" , "Order Region"]
].head(10)

,Order Id,Sales,Customer Segment,Delivery Status,Order Region
160,70534,1500.00000,Consumer,Late delivery,Western Europe
174,68879,999.98999,Consumer,Late delivery,Western Europe
1594,70795,1500.00000,Consumer,Late delivery,Western Europe
5774,70639,1500.00000,Consumer,Late delivery,Western Europe
5836,68773,599.98999,Consumer,Late delivery,Western Europe
7961,70850,1500.00000,Corporate,Late delivery,Western Europe
9603,70786,1500.00000,Consumer,Late delivery,Western Europe
9611,70600,1500.00000,Consumer,Late delivery,Western Europe
12516,70560,1500.00000,Consumer,Late delivery,Western Europe
12528,70609,1500.00000,Consumer,Late delivery,Western Europe


## Business Segmentation with np.select()

In [61]:
# Filtering removes rows from consideration.
# Classification keeps all rows and assigns a business label.
conditions = [
     (df["Sales"] > 1000) ,
     (df["Sales"] > 500)
]
choices = [
     "High_value" , "Medium_value"
]
df["Sales_category"] = np.select(
     conditions , choices ,default="Standard"
)
df["Sales_category"].value_counts()

Sales_category
Standard        179547
Medium_value       515
High_value         457
Name: count, dtype: int64

Profitability Classification

In [62]:
# How many transactions are highly profitable, normally profitable, or loss-making?
conditions = [
     (df["Order Profit Per Order"] > 100) ,
     (df["Order Profit Per Order"] >= 0)
]
choices = [
     "Highly Profitable" ,
     "Normally Profitable"
]
df["Profit_category"] = np.select(
     conditions , choices , default="Loss Making"
)
df["Profit_category"].value_counts()

Profit_category
Normally Profitable    126425
Loss Making             33784
Highly Profitable       20310
Name: count, dtype: int64

Sales × Profit Matrix

In [63]:
# How many transactions fall into each combination of Sales Category and Profit Category?
sales_profit_matrix = pd.crosstab(
     df["Sales_category"] ,
     df["Profit_category"] ,
     margins=True
)
sales_profit_matrix

Profit_category,Highly Profitable,Loss Making,Normally Profitable,All
Sales_category,,,,
High_value,346,76,35,457
Medium_value,319,89,107,515
Standard,19645,33619,126283,179547
All,20310,33784,126425,180519


Investigate High-Value Loss-Making Transactions

In [67]:
# Which high-value transactions are generating losses, and where are they occurring?
high_value_loss = df[
    (df["Sales_category"] == "High_value") &
    (df["Profit_category"] == "Loss Making")
]
high_value_loss.shape

(76, 33)

In [69]:
high_value_loss [
     [
          "Order Id" ,
           "Sales" ,
           "Order Profit Per Order" ,
           "Customer Segment" ,
           "Order Region" ,
           "Category Name" ,
           "Product Name" ,
          "Order Item Discount"
     ]
].sort_values(
     by= "Order Profit Per Order"
).head(10)

,Order Id,Sales,Order Profit Per Order,Customer Segment,Order Region,Category Name,Product Name,Order Item Discount
63990,68859,1999.98999,-4274.97998,Consumer,Northern Europe,Strength Training,SOLE E35 Elliptical,100.0
169249,70533,1500.00000,-3442.50000,Home Office,Southern Europe,Computers,Dell Laptop,150.0
8259,74556,1500.00000,-3366.00000,Corporate,Oceania,Computers,Dell Laptop,180.0
126391,70760,1500.00000,-3000.00000,Consumer,Western Europe,Computers,Dell Laptop,300.0
33845,74508,1500.00000,-2592.00000,Consumer,South Asia,Computers,Dell Laptop,60.0
12516,70560,1500.00000,-2550.00000,Consumer,Western Europe,Computers,Dell Laptop,0.0
84975,70537,1500.00000,-2351.25000,Corporate,Western Europe,Computers,Dell Laptop,75.0
11909,70845,1500.00000,-2328.00000,Home Office,Western Europe,Computers,Dell Laptop,45.0
97532,70753,1500.00000,-2280.00000,Home Office,Western Europe,Computers,Dell Laptop,75.0
82022,70683,1500.00000,-2255.25000,Consumer,Western Europe,Computers,Dell Laptop,45.0


Discount Analysis

In [73]:
#What is the average and total discount associated with high-value loss-making transactions?
high_value_loss["Order Item Discount"].agg(
     ["sum" ,"mean","min" ,"max"]
)

sum     11482.500000
mean      151.085526
min         0.000000
max       375.000000
Name: Order Item Discount, dtype: float64

In [76]:
df["Order Item Discount"].agg(
    ["sum", "mean", "min", "max"]
)

sum     3.730378e+06
mean    2.066474e+01
min     0.000000e+00
max     5.000000e+02
Name: Order Item Discount, dtype: float64

## Discount Category Analysis

Discount = 0          → No Discount
0 < Discount <= 100   → Low Discount
100 < Discount <= 250 → Medium Discount
> 250                 → High Discount

In [78]:
conditions = [
     (df["Order Item Discount"] == 0) ,
     (df["Order Item Discount"] <= 100) ,
     (df["Order Item Discount"] <= 250) ]
choices = [
     "No Discount" ,
     "Low Discount" ,
     "Medium Discount" 
]
df["Discount_category"] = np.select(
     conditions , choices , default="High Discount"
)

In [79]:
df["Discount_category"].value_counts()

Discount_category
Low Discount       169944
No Discount         10028
Medium Discount       440
High Discount         107
Name: count, dtype: int64

Discount × Profit Analysis

In [81]:
# How does profitability vary across different discount levels?
discount_profit_analysis = pd.crosstab(
     df["Discount_category"],
     df["Profit_category"],
     margins=True
)
discount_profit_analysis

Profit_category,Highly Profitable,Loss Making,Normally Profitable,All
Discount_category,,,,
High Discount,77,18,12,107
Low Discount,18522,31848,119574,169944
Medium Discount,268,75,97,440
No Discount,1443,1843,6742,10028
All,20310,33784,126425,180519


Calculate Profitability Rate by Discount Category

In [82]:
# What percentage of transactions in each discount category are loss-making?
loss_rate = (
     discount_profit_analysis["Loss Making"]
     .div(discount_profit_analysis["All"])
     .mul(100)

)

In [83]:
loss_rate = loss_rate.drop("All")

In [84]:
loss_rate.sort_values(
     ascending= False
)

Discount_category
Low Discount       18.740291
No Discount        18.378540
Medium Discount    17.045455
High Discount      16.822430
dtype: float64